# AI응용_4 — Optical Flow 및 추적 알고리즘


**실습 안내:** 문제 코드의 `TODO`를 먼저 작성한 후 아래 정답 셀과 비교하세요. 영상 프레임이나 모델 가중치가 필요한 셀은 원본 실습 자료의 입력 데이터를 준비한 뒤 실행합니다. 실행 결과는 초기화되어 있습니다.

## 학습 목표

- 객체 탐색, 배경 분리, 색상 추적, Sparse/Dense Optical Flow의 차이를 설명한다.
- MeanShift, CamShift, Lucas–Kanade, Farneback, RAFT의 입출력과 한계를 비교한다.
- OpenCV/NumPy/PyTorch의 주요 함수와 배열 차원을 직접 확인한다.

## 기법 비교

| 방식 | 출력 | 주요 함수 |
|---|---|---|
| 템플릿 매칭 | 유사도 맵·일치 좌표 | `matchTemplate`, `minMaxLoc` |
| MOG2 | 전경 마스크 | `createBackgroundSubtractorMOG2` |
| MeanShift | 고정 크기 추적 윈도우 | `meanShift` |
| CamShift | 회전·크기 적응 윈도우 | `CamShift` |
| Sparse LK | 특징점별 이동 | `calcOpticalFlowPyrLK` |
| Dense Farneback | 픽셀별 `(dx,dy)` | `calcOpticalFlowFarneback` |
| RAFT | 반복 정제된 Dense Flow | `raft_large`, `raft_small` |

## 자료별 핵심 정리와 복습 포인트

### 강의 PDF — Optical Flow 및 추적 알고리즘

**핵심 내용:** 템플릿 매칭(3~5쪽), 특징자 추출·매칭(6~9쪽), MeanShift(10~11쪽), CamShift(12~13쪽), Lucas–Kanade(14~15쪽), Dense Flow(16~17쪽), RAFT(18쪽), 종합 비교(19쪽).

**복습 포인트:** 매칭 방법별 최소·최대값 선택, Keypoint와 Descriptor 구별, 각 방식의 출력 차이. `TM_CCOEFF_NORMED`는 음수도 가능하다는 점에 주의.

### 01 특징자와 매칭.ipynb

**핵심 내용:** 그레이스케일 변환 후 `matchTemplate(TM_CCOEFF_NORMED)`와 `minMaxLoc`으로 템플릿 위치를 찾고 사각형을 표시합니다. `np.where`로 임계값 이상의 후보도 확인합니다.

**복습 포인트:** 이 파일의 중심 실습은 특징자 기술자 매칭이 아닌 템플릿 매칭입니다. `shape[::-1]`과 `np.where`의 y,x 순서를 확인.

### 02 background.ipynb

**핵심 내용:** `createBackgroundSubtractorMOG2`로 프레임별 전경 마스크를 생성합니다.

**복습 포인트:** 전경 마스크는 객체의 고유 ID가 아닙니다. 조명 변화·그림자·배경 학습 시간의 영향을 설명.

### 03 MeanShift 트래킹.ipynb

**핵심 내용:** HSV ROI → Hue 히스토그램(`calcHist`) → 정규화 → 역투영(`calcBackProject`) → `meanShift` 순서로 색상 객체를 추적합니다.

**복습 포인트:** `frame[y:y+h,x:x+w]`와 `term_crit`의 역할, 고정 크기 윈도우의 한계.

### 04 CamShift 트래킹.ipynb

**핵심 내용:** 색상 역투영 후 `CamShift`로 위치와 회전 사각형을 얻고 `boxPoints`로 꼭짓점을 표시합니다.

**복습 포인트:** MeanShift와 달리 크기·각도에 적응하지만 유사 색상 배경에는 취약. 반환값 두 개의 의미.

### 05 optical_flow_sparse.ipynb

**핵심 내용:** `goodFeaturesToTrack`과 `calcOpticalFlowPyrLK`로 코너 점을 추적하고 이동 궤적을 표시합니다.

**복습 포인트:** `p1,st,err`, `st==1`, `winSize`, `maxLevel`을 설명.

### 06 Lucas_Kanade.ipynb

**핵심 내용:** 첫 프레임 특징점을 피라미드 Lucas–Kanade로 추적합니다. 05번과 주제가 중복되는 실습입니다.

**복습 포인트:** 밝기 보존, 작은 이동, 국소 운동 일치의 세 가정과 피라미드 효과.

### 07 optical_flow_dense_livecoding.ipynb

**핵심 내용:** `np.stack`으로 좌표를 구성하고 `calcOpticalFlowFarneback`으로 픽셀별 광류를 계산해 화살표로 시각화합니다.

**복습 포인트:** `flow.shape=(H,W,2)`, `flow[y,x]=[dx,dy]`, `np.stack(axis=1)`의 의미.

### 08 RAFT_opticalFlow.ipynb

**핵심 내용:** Torchvision `raft_large`/`raft_small` 사전학습 모델을 사용해 두 프레임에서 반복별 Flow를 얻고 마지막 Flow를 시각화·비교합니다.

**복습 포인트:** 입력 `(1,3,H,W)`, 출력 `(1,2,H,W)`, All-Pairs Correlation, GRU 반복 보정, GPU·가중치 다운로드 필요.

## 그룹스터디 풀이 문제

**16문항 중 코드 작성 13문항, 개념 3문항.** 각 문제의 TODO를 먼저 작성하고 정답 셀과 비교하세요. 필요한 변수(`gray`, `frame` 등)는 원본 실습에서 로드한 영상을 사용합니다.

### 문제 01 · 코드 · 템플릿 매칭 최적 위치 찾기

**문제:** 원본 gray와 template을 이용해 정규화 상관계수 매칭 후 최적 좌상단 좌표를 구하세요.

**시작 코드 (직접 작성)**

In [ ]:
result = cv2.matchTemplate(gray, template, cv2.TM_CCOEFF_NORMED)
# TODO: 최적 위치 구하기

**정답 코드 — 직접 풀이 후 확인**

In [ ]:
_, max_val, _, max_loc = cv2.minMaxLoc(result)
top_left = max_loc
print(top_left, max_val)

### 문제 02 · 코드 · SQDIFF 매칭 영역 표시

**문제:** TM_SQDIFF_NORMED의 최적 위치에 템플릿 크기의 사각형을 그리세요.

**시작 코드 (직접 작성)**

In [ ]:
result = cv2.matchTemplate(gray, template, cv2.TM_SQDIFF_NORMED)
# TODO: 위치, 너비/높이, 사각형

**정답 코드 — 직접 풀이 후 확인**

In [ ]:
min_val, _, min_loc, _ = cv2.minMaxLoc(result)
h, w = template.shape[:2]
x, y = min_loc
marked = cv2.rectangle(image.copy(), (x,y), (x+w,y+h), (0,255,0), 2)

### 문제 03 · 코드 · MOG2 전경 추출

**문제:** 프레임에 대해 전경 마스크를 얻고 전경 픽셀만 남기세요.

**시작 코드 (직접 작성)**

In [ ]:
# TODO: 배경 모델 생성과 마스크 적용

**정답 코드 — 직접 풀이 후 확인**

In [ ]:
bg = cv2.createBackgroundSubtractorMOG2()
fg_mask = bg.apply(frame)
foreground = cv2.bitwise_and(frame, frame, mask=fg_mask)
# 실제 동영상에서는 bg를 루프 밖에서 생성

### 문제 04 · 코드 · ROI 히스토그램

**문제:** ROI의 Hue 히스토그램을 만들고 정규화하세요.

**시작 코드 (직접 작성)**

In [ ]:
roi = frame[y:y+h, x:x+w]
# TODO: HSV 변환, Hue 히스토그램, 정규화

**정답 코드 — 직접 풀이 후 확인**

In [ ]:
hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
roi_hist = cv2.calcHist([hsv_roi], [0], None, [180], [0,180])
cv2.normalize(roi_hist, roi_hist, 0, 255, cv2.NORM_MINMAX)

### 문제 05 · 코드 · MeanShift 한 프레임 갱신

**문제:** 역투영 후 meanShift로 추적 윈도우를 갱신하세요.

**시작 코드 (직접 작성)**

In [ ]:
term_crit = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 1)
# TODO

**정답 코드 — 직접 풀이 후 확인**

In [ ]:
backproj = cv2.calcBackProject([hsv_frame], [0], roi_hist, [0,180], 1)
_, track_window = cv2.meanShift(backproj, track_window, term_crit)

### 문제 06 · 코드 · CamShift 회전 사각형

**문제:** CamShift 실행 후 회전 사각형 꼭짓점을 구하세요.

**시작 코드 (직접 작성)**

In [ ]:
# TODO: CamShift 및 boxPoints

**정답 코드 — 직접 풀이 후 확인**

In [ ]:
rotated_rect, track_window = cv2.CamShift(backproj, track_window, term_crit)
pts = np.int32(cv2.boxPoints(rotated_rect))

### 문제 07 · 코드 · 코너 특징점 검출

**문제:** old_gray에서 최대 100개 코너를 검출하세요.

**시작 코드 (직접 작성)**

In [ ]:
# TODO: goodFeaturesToTrack

**정답 코드 — 직접 풀이 후 확인**

In [ ]:
p0 = cv2.goodFeaturesToTrack(old_gray, maxCorners=100, qualityLevel=0.3, minDistance=7, blockSize=7)

### 문제 08 · 코드 · Sparse LK 유효점 추출

**문제:** LK 광류 계산 후 유효한 점의 이전·현재 좌표만 남기세요.

**시작 코드 (직접 작성)**

In [ ]:
# TODO: calcOpticalFlowPyrLK, st 필터

**정답 코드 — 직접 풀이 후 확인**

In [ ]:
p1, st, err = cv2.calcOpticalFlowPyrLK(old_gray, gray, p0, None, winSize=(15,15), maxLevel=2)
valid = st.ravel() == 1
good_new = p1[valid]
good_old = p0[valid]

### 문제 09 · 코드 · LK 이동량 계산

**문제:** 유효한 특징점의 평균 이동 거리를 계산하세요.

**시작 코드 (직접 작성)**

In [ ]:
# TODO: good_old와 good_new 사용

**정답 코드 — 직접 풀이 후 확인**

In [ ]:
displacement = good_new - good_old
distance = np.linalg.norm(displacement, axis=1)
mean_distance = distance.mean() if len(distance) else 0.0

### 문제 10 · 코드 · Dense Flow 벡터 추출

**문제:** flow의 (x=100,y=50) 위치에서 dx,dy와 크기를 구하세요.

**시작 코드 (직접 작성)**

In [ ]:
# TODO: 인덱싱과 norm

**정답 코드 — 직접 풀이 후 확인**

In [ ]:
dx, dy = flow[50,100]
magnitude = float(np.hypot(dx,dy))

### 문제 11 · 코드 · Farneback Dense Flow

**문제:** 두 회색조 프레임으로 Farneback 광류를 계산하세요.

**시작 코드 (직접 작성)**

In [ ]:
# TODO: calcOpticalFlowFarneback

**정답 코드 — 직접 풀이 후 확인**

In [ ]:
flow = cv2.calcOpticalFlowFarneback(old_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
print(flow.shape)

### 문제 12 · 코드 · 좌표 배열 구성

**문제:** x_idx, y_idx로 (N,2) 좌표 배열을 만드세요.

**시작 코드 (직접 작성)**

In [ ]:
# TODO: np.stack

**정답 코드 — 직접 풀이 후 확인**

In [ ]:
points = np.stack((x_idx, y_idx), axis=1)
print(points.shape)

### 문제 13 · 코드 · RAFT 최종 Flow 추출

**문제:** RAFT 출력 리스트의 마지막 결과를 (H,W,2)로 바꾸세요.

**시작 코드 (직접 작성)**

In [ ]:
# TODO: 마지막 결과와 shape

**정답 코드 — 직접 풀이 후 확인**

In [ ]:
final_flow = list_of_flows[-1]
flow_chw = final_flow[0]
flow_hwc = flow_chw.permute(1,2,0)

### 문제 14 · 개념 · 추적과 전경 분리 구별

**질문:** MOG2의 출력만으로 객체의 동일 ID를 유지할 수 없는 이유를 설명하세요.

**정답·해설:** MOG2는 전경 마스크만 생성하므로 객체별 ID 부여와 프레임 간 데이터 연관이 필요합니다.

### 문제 15 · 개념 · MeanShift와 CamShift 비교

**질문:** 크기와 방향 변화 시 CamShift가 유리한 이유와 공통 한계를 설명하세요.

**정답·해설:** CamShift는 윈도우 크기와 방향에 적응합니다. 두 방식 모두 유사 색상 배경에 취약합니다.

### 문제 16 · 개념 · Lucas–Kanade 가정

**질문:** 세 가지 가정과 큰 이동 대응책을 설명하세요.

**정답·해설:** 밝기 보존, 작은 이동, 국소 운동 일치. 피라미드 수준과 윈도우 크기를 조정합니다.

## 그룹 실습 미션 — 코드 작성형

아래 3개 미션은 노션 페이지와 동일한 순서입니다. **TODO 셀을 먼저 작성**하고, 다음 정답 코드 셀을 비교하세요. 실제 영상 프레임과 모델 입력 텐서는 원본 노트북에서 준비해야 합니다.

### 미션 A · MeanShift와 CamShift 추적 비교

**문제:** 동일한 BGR 프레임 `frame0`, `frame1`, 초기 ROI `roi_rect=(x,y,w,h)`를 이용하세요.

**TODO — 직접 작성**

In [ ]:
import cv2
import numpy as np
x, y, w, h = roi_rect
roi = frame0[y:y+h, x:x+w]
# TODO 1: ROI HSV 변환, Hue 히스토그램 생성·정규화
# TODO 2: frame1에서 역투영
# TODO 3: MeanShift/CamShift 실행
# TODO 4: 결과 출력

**정답 코드 — TODO 완성 후 확인**

In [ ]:
hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
hist = cv2.calcHist([hsv_roi], [0], None, [180], [0, 180])
cv2.normalize(hist, hist, 0, 255, cv2.NORM_MINMAX)
hsv1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2HSV)
backproj = cv2.calcBackProject([hsv1], [0], hist, [0, 180], 1)
term = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 1)
_, mean_window = cv2.meanShift(backproj, roi_rect, term)
rot_rect, cam_window = cv2.CamShift(backproj, roi_rect, term)
cam_points = np.int32(cv2.boxPoints(rot_rect))
print('MeanShift:', mean_window)
print('CamShift:', cam_window, cam_points)

**검증 및 토론:** 크기·회전 변화에 따른 차이를 비교하고, 비슷한 배경색에서 실패할 가능성을 토론하세요.

### 미션 B · Sparse LK와 Dense Farneback 비교

**문제:** 동일 크기 회색조 프레임 `old_gray`, `gray`에서 Sparse/Dense 이동을 비교하세요.

**TODO — 직접 작성**

In [ ]:
import cv2
import numpy as np
# TODO 1: goodFeaturesToTrack으로 p0 생성
# TODO 2: LK 추적 및 st==1 필터
# TODO 3: Sparse 평균 이동 크기
# TODO 4: Farneback Flow 및 Dense 평균 이동 크기

**정답 코드 — TODO 완성 후 확인**

In [ ]:
p0 = cv2.goodFeaturesToTrack(old_gray, maxCorners=100, qualityLevel=0.3, minDistance=7)
if p0 is None:
    raise ValueError('추적할 특징점이 없습니다.')
p1, st, err = cv2.calcOpticalFlowPyrLK(old_gray, gray, p0, None, winSize=(15, 15), maxLevel=2)
valid = st.ravel() == 1
sparse_delta = p1[valid].reshape(-1, 2) - p0[valid].reshape(-1, 2)
sparse_mean = float(np.linalg.norm(sparse_delta, axis=1).mean()) if len(sparse_delta) else 0.0
flow = cv2.calcOpticalFlowFarneback(old_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
dense_mean = float(np.linalg.norm(flow, axis=2).mean())
print('Sparse valid:', valid.sum(), 'mean:', sparse_mean)
print('Dense shape:', flow.shape, 'mean:', dense_mean)

**검증 및 토론:** Sparse는 특징점 평균, Dense는 전체 픽셀 평균이므로 평균값을 그대로 정확도 순위로 해석하지 마세요.

### 미션 C · RAFT-Large와 Small 추론 비교

**문제:** 전처리된 `(1,3,H,W)` 텐서 `img1`, `img2`로 두 모델의 최종 Flow와 시간을 비교하세요.

**TODO — 직접 작성**

In [ ]:
import time
import torch
from torchvision.models.optical_flow import raft_small, raft_large, Raft_Small_Weights, Raft_Large_Weights
# TODO 1: 두 사전학습 모델 생성 및 eval
# TODO 2: no_grad에서 추론 시간 측정
# TODO 3: 최종 Flow shape 출력

**정답 코드 — TODO 완성 후 확인**

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
models = {
    'small': raft_small(weights=Raft_Small_Weights.DEFAULT).to(device).eval(),
    'large': raft_large(weights=Raft_Large_Weights.DEFAULT).to(device).eval(),
}
with torch.no_grad():
    for name, model in models.items():
        if device.type == 'cuda': torch.cuda.synchronize()
        start = time.perf_counter()
        flows = model(img1.to(device), img2.to(device))
        if device.type == 'cuda': torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        final_flow = flows[-1]
        print(name, final_flow.shape, f'{elapsed:.3f}s', 'iterations:', len(flows))

**검증 및 토론:** 동일한 장치·해상도·워밍업 조건으로 비교하세요. 사전학습 가중치 다운로드와 충분한 메모리가 필요합니다.

## 최종 복습 체크리스트

- [ ] Keypoint와 Descriptor를 구별한다.
- [ ] MOG2의 전경 분리와 ID 추적을 구별한다.
- [ ] MeanShift와 CamShift의 윈도우 차이를 설명한다.
- [ ] Lucas–Kanade의 세 가지 가정을 설명한다.
- [ ] `p1, st, err`와 `(H,W,2)`를 해석한다.
- [ ] RAFT의 Feature Encoder → Correlation → GRU 업데이트를 설명한다.

**주의:** 코드 정답은 해당 변수가 사전에 정의되었다는 전제의 실습 예시입니다. 학습 데이터 로딩은 원본 노트북의 경로·환경을 따라 준비하세요.